In [ ]:
pip install pyspark

In [ ]:
# Loading the dataset into colab
from google.colab import files
import zipfile
import os

# Uploading the ZIP file manually
uploaded = files.upload()

# Extracting the ZIP file
zip_file_name = "hospitals_current_data.zip"
extract_folder = "/content/dataset"  # Destination folder which uploads all files of the database

with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)

print("Files extracted successfully!")


Saving hospitals_current_data.zip to hospitals_current_data.zip
Files extracted successfully!


In [ ]:
from pyspark.sql import SparkSession

# Starting a Spark session
spark = SparkSession.builder.appName("HealthcareDataCleaning").getOrCreate()

# Loading all CSV files from the extracted folder with all datasets
df = spark.read.csv("/content/dataset/*.csv", header=True, inferSchema=True)

# Testing to show if the first 5 rows loaded correctly
df.show(5)

+-----------+--------------------+--------------------+---------+-----+--------+-------------+----------------+--------------------+--------------------+-------------------------+--------------------------+-----------------------------------+---------------------+------------------------------+------------------------+---------------------------+------------------------------------+----------------------------+-------------------------------------+----------+----------+
|Facility ID|       Facility Name|             Address|City/Town|State|ZIP Code|County/Parish|Telephone Number|   HCAHPS Measure ID|     HCAHPS Question|HCAHPS Answer Description|Patient Survey Star Rating|Patient Survey Star Rating Footnote|HCAHPS Answer Percent|HCAHPS Answer Percent Footnote|HCAHPS Linear Mean Value|Number of Completed Surveys|Number of Completed Surveys Footnote|Survey Response Rate Percent|Survey Response Rate Percent Footnote|Start Date|  End Date|
+-----------+--------------------+----------------

In [ ]:
# before cleaning we have to identify missing values, duplicate rows and incorrect data types
df.printSchema() # To check and diplay the column names and data types of the dataset

from pyspark.sql.functions import col, when, count
# helps to count the number of missing values in each column

df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()
# count(when(col(c).isNull(),c)) checks the number of missing values
# .aliac(c) renames the output column with the actual name of the column
# .show() displays the missing value count (final output)

root
 |-- Facility ID: string (nullable = true)
 |-- Facility Name: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- City/Town: string (nullable = true)
 |-- State: string (nullable = true)
 |-- ZIP Code: string (nullable = true)
 |-- County/Parish: string (nullable = true)
 |-- Telephone Number: string (nullable = true)
 |-- HCAHPS Measure ID: string (nullable = true)
 |-- HCAHPS Question: string (nullable = true)
 |-- HCAHPS Answer Description: string (nullable = true)
 |-- Patient Survey Star Rating: string (nullable = true)
 |-- Patient Survey Star Rating Footnote: string (nullable = true)
 |-- HCAHPS Answer Percent: string (nullable = true)
 |-- HCAHPS Answer Percent Footnote: string (nullable = true)
 |-- HCAHPS Linear Mean Value: string (nullable = true)
 |-- Number of Completed Surveys: string (nullable = true)
 |-- Number of Completed Surveys Footnote: string (nullable = true)
 |-- Survey Response Rate Percent: string (nullable = true)
 |-- Survey Response 

In [ ]:
# we still have lots of missing values and incorrect data type problems

from pyspark.sql.functions import col, when, count, mean, lower

# Converting Numeric Columns from String to Integer/Float
numeric_columns = ["ZIP Code", "HCAHPS Answer Percent", "HCAHPS Linear Mean Value",
                   "Number of Completed Surveys", "Survey Response Rate Percent"]

for col_name in numeric_columns:
    df = df.withColumn(col_name, col(col_name).cast("double"))
    # To Convert to double

# Handling Missing Values
# Fill categorical columns with "Unknown"
categorical_columns = ["Facility Name", "City/Town", "State", "County/Parish"]
for col_name in categorical_columns:
    df = df.fillna({col_name: "Unknown"})

# Fill numeric columns with their mean
for col_name in numeric_columns:
    mean_value = df.select(mean(col_name)).collect()[0][0]  # To calculate mean
    if mean_value is not None:  # Avoid errors if column is empty
        df = df.fillna({col_name: mean_value})

# Standardizing Text Columns (Converting to lowercase)
df = df.withColumn("Facility Name", lower(col("Facility Name")))
df = df.withColumn("City/Town", lower(col("City/Town")))
df = df.withColumn("State", lower(col("State")))

# Removing Duplicate Rows if any
df = df.dropDuplicates()

# To show the cleaned data
df.show(20)


+-----------+--------------------+--------------------+----------------+-----+--------+--------------+----------------+-----------------+--------------------+-------------------------+--------------------------+-----------------------------------+---------------------+------------------------------+------------------------+---------------------------+------------------------------------+----------------------------+-------------------------------------+----------+----------+
|Facility ID|       Facility Name|             Address|       City/Town|State|ZIP Code| County/Parish|Telephone Number|HCAHPS Measure ID|     HCAHPS Question|HCAHPS Answer Description|Patient Survey Star Rating|Patient Survey Star Rating Footnote|HCAHPS Answer Percent|HCAHPS Answer Percent Footnote|HCAHPS Linear Mean Value|Number of Completed Surveys|Number of Completed Surveys Footnote|Survey Response Rate Percent|Survey Response Rate Percent Footnote|Start Date|  End Date|
+-----------+--------------------+------

In [ ]:
# Start date column error fixing (incorrect format of date)
from pyspark.sql.functions import to_date

# Converting "Start Date" to proper date format (assuming it's MM/DD/YYYY or needs conversion)
df = df.withColumn("Start Date", to_date(col("Start Date"), "MM/dd/yyyy"))

# Testing to show fixed data
df.select("Start Date").distinct().show(20)

#saving the changes
df.printSchema()
df.select("Start Date").distinct().show(60)



+----------+
|Start Date|
+----------+
|2023-04-01|
|2020-07-01|
|2024-03-31|
|      NULL|
+----------+

root
 |-- Facility ID: string (nullable = true)
 |-- Facility Name: string (nullable = false)
 |-- Address: string (nullable = true)
 |-- City/Town: string (nullable = false)
 |-- State: string (nullable = false)
 |-- ZIP Code: double (nullable = false)
 |-- County/Parish: string (nullable = false)
 |-- Telephone Number: string (nullable = true)
 |-- HCAHPS Measure ID: string (nullable = true)
 |-- HCAHPS Question: string (nullable = true)
 |-- HCAHPS Answer Description: string (nullable = true)
 |-- Patient Survey Star Rating: string (nullable = true)
 |-- Patient Survey Star Rating Footnote: string (nullable = true)
 |-- HCAHPS Answer Percent: double (nullable = false)
 |-- HCAHPS Answer Percent Footnote: string (nullable = true)
 |-- HCAHPS Linear Mean Value: double (nullable = false)
 |-- Number of Completed Surveys: double (nullable = false)
 |-- Number of Completed Surveys Foo

In [ ]:
# test to check number of null values in the column
df.select(count(when(col("Start Date").isNull(), "Start Date"))).show()


+---------------------------------------------------------+
|count(CASE WHEN (Start Date IS NULL) THEN Start Date END)|
+---------------------------------------------------------+
|                                                   785058|
+---------------------------------------------------------+



In [ ]:
# Final checking and modifications before saving the cleaned data
from pyspark.sql.functions import col, when ,count, mean, lower

# Checking schema to ensure correct data types
df.printSchema()
# showed double for zipcode before running the cell again

# Checking missing values after previous cleaning
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

# Since ZIP code is shown in decimal, to ensure correct format
df = df.withColumn("ZIP Code", col("ZIP Code").cast("int"))

# Remove invalid ZIP codes (should be between 10000 and 99999)
df = df.filter((col("ZIP Code") >= 10000) & (col("ZIP Code") <= 99999))
df.printSchema()

# Show unique ZIP codes again to verify
df.select("ZIP Code").distinct().show(20)

# Final check, first 5 rows
df.show(5)

# Last step : To Save Cleaned Data
import shutil

# Delete the existing folder (if it exists) to avoid errors
shutil.rmtree("cleaned_data.csv", ignore_errors=True)

# Save the cleaned data, ensuring overwrite mode
df.write.mode("overwrite").csv("cleaned_data.csv", header=True)
print("Cleaned data saved successfully! ")

root
 |-- Facility ID: string (nullable = true)
 |-- Facility Name: string (nullable = false)
 |-- Address: string (nullable = true)
 |-- City/Town: string (nullable = false)
 |-- State: string (nullable = false)
 |-- ZIP Code: integer (nullable = true)
 |-- County/Parish: string (nullable = false)
 |-- Telephone Number: string (nullable = true)
 |-- HCAHPS Measure ID: string (nullable = true)
 |-- HCAHPS Question: string (nullable = true)
 |-- HCAHPS Answer Description: string (nullable = true)
 |-- Patient Survey Star Rating: string (nullable = true)
 |-- Patient Survey Star Rating Footnote: string (nullable = true)
 |-- HCAHPS Answer Percent: double (nullable = false)
 |-- HCAHPS Answer Percent Footnote: string (nullable = true)
 |-- HCAHPS Linear Mean Value: double (nullable = false)
 |-- Number of Completed Surveys: double (nullable = false)
 |-- Number of Completed Surveys Footnote: string (nullable = true)
 |-- Survey Response Rate Percent: double (nullable = false)
 |-- Survey 

In [ ]:
# downloading the final cleaned data file
!cat cleaned_data.csv/*.csv > cleaned_data_final.csv
from google.colab import files
files.download("cleaned_data_final.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>